In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('D:\GITHUB\LinkedIn Hiring Posts - Feb 2026(n8n Automation)\LinkedIn Hiring Posts - Feb 2026(n8n Automation)\ecommerce_messy_data.csv')

In [3]:
df.head()

,order_id,customer_name,email,phone,street_address,city,state,zip_code,product_name,category,subcategory,unit_price,quantity,discount_%,total_amount,shipping_cost,order_date,delivery_date,payment_method,order_status,customer_rating,review_count
0,306618,Patricia Brown,patricia.brown@gmail.com,+12462613434,8177 Cedar Dr,PHILADELPHIA,PA,55673,PRO SHOES,clothing,Shoes,509.28,6.0,21.2,2407.88,4.36,10/26/2022,11/17/2022,Debit Card,Cancelled,1.9,485
1,204530,James Smith,james.smith@gmail.com,5209519670,4308 Main Blvd,Phoenix,AZ,86569-9398,Car Parts Plus,AUTOMOTIVE,Car Parts,534.40,7.0,24.8,2813.08,0.94,NaN,2023-06-27,Credit Card,Shipped,4.7,2559
2,ord618933,MICHAEL BROWN,NaN,NaN,4857 Elm Ln,San Antonio,TX,26967-3096,ultra skincare x,Beauty,Skincare,153.47,4.0,7.2,569.68,2.98,NaN,-,debit card,Processing,4.2,1398
3,571689,Patricia Smith,patricia@yahoo.com,(499) 781-7874,8724 Elm Ave,New York,NY,99267,Basic Water Sports,Sports,Water Sports,850.18,1.0,NaN,850.18,14.45,2022-09-20,2022-10-14,Google Pay,Cancelled,1.2,-
4,ord197194,Linda Garcia,linda.garcia@gmail.com,(861) 155-4625,9545 Main Ave,Dallas,TX,NaN,Basic International Max,Food,International,382.78,5.0,-,1913.90,0.87,2024-12-25,01/15/2025,Bank Transfer,Returned,3.5,846


In [5]:
df.columns

Index(['order_id', 'customer_name', 'email', 'phone', 'street_address', 'city',
       'state', 'zip_code', 'product_name', 'category', 'subcategory',
       'unit_price', 'quantity', 'discount_%', 'total_amount', 'shipping_cost',
       'order_date', 'delivery_date', 'payment_method', 'order_status',
       'customer_rating', 'review_count'],
      dtype='object')

# ID column

#### Exploring the ID Column
    1. Explored the columns[ Checked for nulls and duplicate values]
    2. Standardized the column[Removed all the alphabets]
    3. Handled the nulls and duplicates
    4. Validated the changes

In [ ]:
## Step 1 — Explore

# Check how many nulls exist
print("Number of null values in each column:")
print(df.isnull().sum())

Number of null values in each column:
order_id             30
customer_name        30
email              1393
phone              1507
street_address      723
city                282
state               358
zip_code           1337
product_name         35
category             30
subcategory         588
unit_price           30
quantity             30
discount_%          902
total_amount         30
shipping_cost       700
order_date          995
delivery_date       927
payment_method      348
order_status        350
customer_rating    1642
review_count       1260
dtype: int64


In [5]:
# Check how many duplicates exist
print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 29


In [6]:
## Step 2 — Standardize

# Remove all prefixes and non-numeric characters (assumption documented — client confirmed prefixes carry no meaning)
df['order_id'] = df['order_id'].str.replace(r'[^\d]', '', regex=True)
# Keep the column as string type
df['order_id'] = df['order_id'].astype("string") # Using pandas string type to preserve NULL values

In [7]:
## Step 3 — Handle Nulls

# Drop rows where order_id is null or blank after standardization
original_count = len(df)
df = df[df['order_id'].notna() & (df['order_id'] != '')]
new_count = len(df)
print(f"Rows dropped: {original_count - new_count}")

Rows dropped: 30


In [8]:
## Step 4 — Handle Duplicates

# Identical rows → keep first occurrence, drop the rest
df.drop_duplicates(inplace=True)
# Same ID but different row data → flag them, do not delete, escalate to client
duplicate_ids = df[df.duplicated(subset=['order_id'], keep=False)]['order_id'].unique()
print("Duplicate order IDs:", duplicate_ids)

Duplicate order IDs: <StringArray>
['367762', '903603', '127560', '193108', '138760', '790306', '627761',
 '936923', '335513', '520538',
 ...
 '612349', '555910', '768124', '231185', '812661', '567778', '182919',
 '464361', '974151', '786660']
Length: 375, dtype: string


In [33]:
## Step 5 — Validate

# #Confirm zero non-numeric characters remain
print("Unique characters in order_id after cleaning:", df['order_id'].apply(lambda x: set(x)).explode().unique())
#Compare duplicate count before and after
print("Number of duplicate rows after cleaning:", df.duplicated().sum())
#Compare total row count before and after — explain any significant drop
print(f"Total rows before cleaning: {original_count}, after cleaning: {len(df)}")

Unique characters in order_id after cleaning: ['0' '6' '1' '8' '3' '5' '2' '4' '9' '7']
Number of duplicate rows after cleaning: 0
Total rows before cleaning: 10000, after cleaning: 9970


# Order Date

#### Exploring the order_date Column
    1. Explored the columns[ Checked for nulls]
    2. Find out all the different formats in the order_date column
    3. Handled inconsistency in the date
    4. Validated the changes

###

In [15]:
# Find out all the null characters in the order_date column
print("Number of null values in the dataframe:  ",df["order_date"].isnull().sum())

Number of null values in the dataframe:   965


In [16]:
# Find out all the different formats in the order_date column
print("Unique formats in order_date column:")
df["length"] = df["order_date"].str.len()
print(df.groupby("length").size())

Unique formats in order_date column:
length
1.0       49
10.0    7960
12.0      89
13.0     153
14.0     155
15.0      88
16.0     177
17.0     243
18.0      91
dtype: int64


In [34]:
df['order_date'].unique()

array(['10/26/2022', nan, '2022-09-20', ..., '2022-11-03',
       'November 18, 2022', '09-11-2022'], dtype=object)

In [ ]:
from dateutil import parser
import pandas as pd

def parse_date_safe(val):
    if pd.isna(val) or str(val).strip() in ('', 'nan', 'None'):
        return pd.NaT
    try:
        # dayfirst=False ensures MM/DD/YYYY takes priority over DD/MM/YYYY
        # where ambiguous (e.g. 10/06/2022 → Oct 6, not June 10)
        return parser.parse(str(val).strip(), dayfirst=False)
    except (ValueError, OverflowError):
        return pd.NaT

df['order_date_cleaned'] = (
    df['order_date']
    .astype(str)
    .str.strip()
    .str.replace(r'\xa0', '', regex=True)         # Removes non-breaking spaces — invisible characters that sometimes sneak in from websites or Excel
    .str.replace(r'^-$', '', regex=True)          # If a cell contains just a dash ("-"), replace it with empty string. A lone dash usually means "no data"
    .map(parse_date_safe)                         # Runs our safe function on every single value to convert text → date
    .dt.normalize()                                # strips time component
    .dt.strftime('%Y-%m-%d')                       # final yyyy-mm-dd string
)

df['order_date_cleaned'] = pd.to_datetime(df['order_date_cleaned'])

In [29]:
df[['order_date', 'order_date_cleaned']].head(10)

,order_date,order_date_cleaned
0,10/26/2022,2022-10-26
1,NaN,NaT
2,NaN,NaT
3,2022-09-20,2022-09-20
4,2024-12-25,2024-12-25
5,04/16/2024,2024-04-16
6,2024-12-25,2024-12-25
7,NaN,NaT
8,2024-04-10,2024-04-10
9,2022-03-04,2022-03-04


In [37]:
# Check for mismatches — rows where original had a value but cleaned is NaT
mismatches = df[df['order_date'].notna() & df['order_date_cleaned'].isna()]
print(f"Rows with original value but failed cleaning: {len(mismatches)}")
if len(mismatches) > 0:
    print(mismatches['order_date'].unique())

# Date range sanity check — are dates within a reasonable range?
print("Min date:", df['order_date_cleaned'].min())
print("Max date:", df['order_date_cleaned'].max())

# Befor and After null counts
print("Num. of null values in order_date column", df['order_date'].isna().sum())
print("Num. of null values in order_date_cleaned column", df['order_date_cleaned'].isna().sum())

Rows with original value but failed cleaning: 800
['13/32/2023' '-']
Min date: 2022-01-01 00:00:00
Max date: 2024-12-31 00:00:00
Num. of null values in order_date column 965
Num. of null values in order_date_cleaned column 1765


In [45]:
df[df['order_date_cleaned'].isna()]['order_date'].value_counts()

13/32/2023    751
-              49
Name: order_date, dtype: int64

In [ ]:
df[df['order_date'].isna() & df['order_date_cleaned'].notna()]

Series([], Name: order_date_cleaned, dtype: int64)

In [83]:
print("Null Values in order_date",df['order_date'].isna().value_counts())
print("-----------------------------------------------------------------------")
print("Null values in order_Date_cleaned", df['order_date_cleaned'].isna().value_counts())

Null Values in order_date False    9005
True      965
Name: order_date, dtype: int64
-----------------------------------------------------------------------
Null values in order_Date_cleaned False    8205
True     1765
Name: order_date_cleaned, dtype: int64


In [85]:
# 8205-9005
1765-965

800

In [89]:
df[df['order_date'].isnull()]['order_date_cleaned'].value_counts()

Series([], Name: order_date_cleaned, dtype: int64)

In [95]:
df[(df['order_date'].isnull()) & (df['order_date_cleaned'] == '')]

,order_id,customer_name,email,phone,street_address,city,state,zip_code,product_name,category,subcategory,unit_price,quantity,discount_%,total_amount,shipping_cost,order_date,delivery_date,payment_method,order_status,customer_rating,review_count,length,order_date_cleaned


### Transactions Column[Unit Price, Total_amount, discount]
    1. Calculate total_amount_1 = unit_price × quantity
    2. Compare with actual total_amount
    3. If they match — discount is 0%, null simply means no discount was applied
    4.If they don't match — a discount exists, and you can calculate it using the delta

Before you execute this, think about these three scenarios:

***Scenario 1***

**unit_price** is negative or zero — we know this exists in our dataset. What happens to your formula in that case?

***Scenario 2***

total_amount is inconsistent with **unit_price** × quantity even after accounting for **discount** — we know roughly 5% of total amounts were deliberately made inconsistent in this dataset. How do you handle those rows?

***Scenario 3***

Multiple columns in the same row are null simultaneously — for example both **unit_price** and **total_amount** are null. Can you still calculate anything?

## Unit Price

In [98]:
df['unit_price'].describe()

count    9970.000000
mean      461.329078
std       335.735164
min      -995.030000
25%       212.732500
50%       469.280000
75%       741.050000
max       999.900000
Name: unit_price, dtype: float64

In [109]:
(-995.03*9.0)*(1.0-0.09)

-8149.295700000001

In [102]:
df[df['unit_price'] < 0]

,order_id,customer_name,email,phone,street_address,city,state,zip_code,product_name,category,subcategory,unit_price,quantity,discount_%,total_amount,shipping_cost,order_date,delivery_date,payment_method,order_status,customer_rating,review_count,length,order_date_cleaned
19,838887,Robert Miller,robert.miller@gmail.com,NaN,8400 Elm Dr,Philadelphia,PA,23260-6431,ULTRA AUTOMOTIVE MAX,Automotive,NaN,-43.30,7.0,NaN,-303.10,13.6,13/32/2023,NaN,Debit Card,Refunded,NaN,3687,10.0,NaT
79,817721,John Johnson,john.johnson@gmail.com,(763) 338-6636,9742 Elm Ln,San Diego,NaN,33822,Basic Home & Kitchen Max,home & kitchen,-,-995.03,9.0,9.0,-4641.03,16.13,08/31/2022,NaN,Credit Card,Refunded,4.4,4314,10.0,2022-08-31
91,529371,robert garcia,robert@,(620) 177-4965,5994 Cedar Dr,NEW YORK,NY,70764,Premium Books X,Books,NaN,-755.66,3.0,5.4,-2144.56,1.65,08/11/2022,2022-09-08,BANK TRANSFER,Cancelled,3.2,2782,10.0,2022-08-11
107,358674,William Davis,william.davis@gmail.com,576-569-4576,8788 Elm St,Los Angeles,CA,NaN,Basic International Plus,Food,International,-839.55,5.0,27.0,-3064.36,29.96,01/05/2022,2022-02-03,Apple Pay,Refunded,3.9,4005,10.0,2022-01-05
119,365035,LINDA DAVIS,NaN,(498) 787-1470,1541 Elm Dr,Chicago,IL,24531,action figures max,Toys,Action Figures,-361.72,8.0,2.8,-2812.73,22.33,13/32/2023,09/16/2022,Gift Card,Cancelled,2.1,2119,10.0,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9820,453018,Michael Moore,michael.moore@gmail.com,NaN,1147 Main Blvd,houston,TX,80698,Ultra Shoes X,clothing,Shoes,-834.01,5.0,2.3,-4074.14,15.34,02/16/2024,03/07/2024,gift card,Processing,NaN,1772,10.0,2024-02-16
9836,223669,Robert Garcia,robert.garcia@gmail.com,930-208-4248,5700 Main Dr,Phoenix,AZ,44704,Cameras Plus,Electronics,Cameras,-56.78,-10.0,-,567.80,6.66,04/19/2022,2022-05-08,Apple Pay,Refunded,2.7,1720,10.0,2022-04-19
9903,308920,mary jones,mary@,-,3235 Maple Blvd,New York,NY,41997,Ultra Children's X,Books,Children's,-646.58,3.0,15.7,-1635.20,22.38,19-10-2022,2022-10-14,Bank Transfer,Shipped,1.6,4353,10.0,2022-10-19
9964,214768,Robert Johnson,INVALID_EMAIL,303-503-7499,8184 Main St,Dallas,TX,68716,International Max,FOOD,International,-298.71,2.0,8.0,-549.63,23.58,2022-10-29,11/28/2022,Bank Transfer,Pending,3.3,1284,10.0,2022-10-29


In [55]:
# Let us check for any blank values in the order_id column
print("Blank order_id values:", df["order_id"].isna().sum())
#Let us drop any blank values in the order_id column
df.dropna(subset=["order_id"], inplace=True)

#let us check for any non-numeric values in the order_id column
print("Non-numeric order_id values:", df["order_id"].str.contains(r'[^\d]').sum())
# # Let is replace any non-numeric values with blank
df["order_id"] = df["order_id"].str.replace(r'[^\d]', '', regex=True)
 
# Let us convert the order_id column to string type
df["order_id"] = df["order_id"].astype(str)

# Let us check for any duplicates in the order_id column
print("Number of duplicate order_id values:", df["order_id"].duplicated().sum())
# let us drop them if there are any
df.drop_duplicates(subset=["order_id"], inplace=True)

# print("Distinct order_id values:", df["order_id"].nunique())


# print("*" * 50)
# print("Non-numeric order_id values:", df["order_id"].str.contains(r'[^\d]').sum())
# print("Blank order_id values:", df["order_id"].isna().sum())
# print("Number of duplicate order_id values:", df["order_id"].duplicated().sum())
# print("Distinct order_id values:", df["order_id"].nunique())



Blank order_id values: 30
Non-numeric order_id values: 7509


### Customer Name

In [11]:
# make all characters lower‑case and then capitalise the first letter
df['customer_name'] = df['customer_name'].str.title()
# remove leading and trailing spaces
df['customer_name'] = df['customer_name'].str.strip()




### email_id

In [18]:
df[["customer_name", "email"]]

,customer_name,email
0,Patricia Brown,patricia.brown@gmail.com
1,James Smith,james.smith@gmail.com
2,Michael Brown,NaN
3,Patricia Smith,patricia@yahoo.com
4,Linda Garcia,linda.garcia@gmail.com
...,...,...
9995,Linda Williams,linda.williams@gmail.com
9996,Jennifer Williams,jennifer.williams@gmail.com
9997,Robert Johnson,NaN
9998,Robert Miller,robert@yahoo.com


In [15]:
print("Distinct email domains:", df["email"].str.split("@").str[1].value_counts())


Distinct email domains: email
gmail.com    4694
yahoo.com    1959
              930
Name: count, dtype: int64
